Future - August

In [ ]:
import xarray as xr
import numpy as np
import os
import matplotlib.pyplot as plt
from windrose import WindroseAxes
from scipy.stats import wasserstein_distance, ks_2samp
import glob

%matplotlib inline

# --- 1. Configuration ---
MY_BINS = np.arange(0, 21, 3)
MAX_PERCENTAGE = 25
TARGET_HEIGHT = 150
YEARS = range(2040, 2081)
BASE_PATH = "/data/nobackup/boland/experiment2/WRF/run"

def calculate_metgrid_wind(file_list, target_z=TARGET_HEIGHT):
    u_150_all, v_150_all = [], []
    
    # Get center indices from first file
    with xr.open_dataset(file_list[0]) as temp_ds:
        mid_lat_idx = len(temp_ds.south_north) // 2
        mid_lon_idx = len(temp_ds.west_east) // 2

    for f in sorted(file_list):
        with xr.open_dataset(f) as ds:
            # Metgrid: UU, VV, GHT (Height in m)
            u_p = ds['UU'].isel(Time=0, south_north=mid_lat_idx, west_east_stag=mid_lon_idx).values
            v_p = ds['VV'].isel(Time=0, south_north_stag=mid_lat_idx, west_east=mid_lon_idx).values
            z_p = ds['GHT'].isel(Time=0, south_north=mid_lat_idx, west_east=mid_lon_idx).values
            
            # Vertical interpolation to target height
            u_150_all.append(np.interp(target_z, z_p, u_p))
            v_150_all.append(np.interp(target_z, z_p, v_p))
        
    u_arr, v_arr = np.array(u_150_all), np.array(v_150_all)
    speed = np.sqrt(u_arr**2 + v_arr**2)
    # Correct meteorological direction (0 is North)
    direction = (270 - np.rad2deg(np.arctan2(v_arr, u_arr))) % 360
    return speed, direction

# --- 2. Processing & Annual Gallery ---
all_speed, all_direction = [], []
yearly_data = {}

for y in YEARS:
    search_path = os.path.join(BASE_PATH, f"met_em.d01.{y}-08-*.nc")
    files = glob.glob(search_path)
    
    if files:
        print(f"Processing {y}...")
        speed, direction = calculate_metgrid_wind(files)
        yearly_data[y] = {'speed': speed, 'direction': direction}
        all_speed.extend(speed)
        all_direction.extend(direction)
        
        # Individual Annual Plot
        ax = WindroseAxes.from_ax()
        ax.bar(direction, speed, normed=True, bins=MY_BINS, opening=0.8, edgecolor='white')
        ax.set_ylim(0, MAX_PERCENTAGE)
        ax.set_legend(title="Wind Speed (m/s)", loc='lower right', decimal_places=0)
        ax.set_title(f"Year: August {y} | Mean: {np.mean(speed):.2f} m/s")
        plt.show()

# --- 3. Double Statistical Ranking (Vector-Based) ---

# Convert lists to numpy arrays for math
all_speed_arr = np.array(all_speed)
all_dir_arr = np.array(all_direction)

# Calculate "Full Climatology" components ONCE
# We use radians for trig functions
u_all = all_speed_arr * np.cos(np.deg2rad(all_dir_arr))
v_all = all_speed_arr * np.sin(np.deg2rad(all_dir_arr))

ks_scores = {}
emd_scores = {}

for y, data in yearly_data.items():
    s_yr = np.array(data['speed'])
    d_yr = np.array(data['direction'])
    
    # Calculate current year components
    u_yr = s_yr * np.cos(np.deg2rad(d_yr))
    v_yr = s_yr * np.sin(np.deg2rad(d_yr))

    # EMD Test (Wasserstein): Sum of distances for U and V
    emd_u = wasserstein_distance(u_all, u_yr)
    emd_v = wasserstein_distance(v_all, v_yr)
    emd_scores[y] = emd_u + emd_v
    
    # KS Test: Sum of D-statistics for U and V
    ks_u, _ = ks_2samp(u_all, u_yr)
    ks_v, _ = ks_2samp(v_all, v_yr)
    ks_scores[y] = ks_u + ks_v

# Find the years with the minimum discrepancy
best_ks_year = min(ks_scores, key=ks_scores.get)
best_emd_year = min(emd_scores, key=emd_scores.get)

# --- 4. Final Comparison Plots ---
def plot_comparison(winner_year, test_name, data_dict):
    fig = plt.figure(figsize=(14, 6))
    
    # Left Plot: The "Ideal" Distribution (40-year Climatology)
    ax1 = fig.add_subplot(121, projection="windrose")
    ax1.bar(all_direction, all_speed, normed=True, bins=MY_BINS, opening=0.8, edgecolor='white')
    ax1.set_ylim(0, MAX_PERCENTAGE)
    ax1.set_legend(title="Wind Speed (m/s)", loc='lower right', decimal_places=0)
    ax1.set_title(f"Target Climatology (2040 - 2080)", fontsize=12)
    
    # Right Plot: The "Winner" Year
    ax2 = fig.add_subplot(122, projection="windrose")
    ax2.bar(data_dict[winner_year]['direction'], data_dict[winner_year]['speed'], 
            normed=True, bins=MY_BINS, opening=0.8, edgecolor='white')
    ax2.set_ylim(0, MAX_PERCENTAGE)
    ax2.set_legend(title="Wind Speed (m/s)", loc='lower right', decimal_places=0)
    ax2.set_title(f"Most Representative Year ({test_name}): {winner_year}", fontsize=12)
    
    plt.tight_layout()
    plt.show()

print(f"\n--- Results ---\nKS Winner: {best_ks_year}\nEMD Winner: {best_emd_year}")
plot_comparison(best_ks_year, "KS", yearly_data)
plot_comparison(best_emd_year, "EMD", yearly_data)

print("\n" + "="*60)
# Print means for validation
print(f"KS Winner Year ({best_ks_year}) Mean Speed: {np.mean(yearly_data[best_ks_year]['speed']):.2f} m/s")
print(f"EMD Winner Year ({best_emd_year}) Mean Speed: {np.mean(yearly_data[best_emd_year]['speed']):.2f} m/s")
print(f"40-Year Climatology Mean Speed: {np.mean(all_speed):.2f} m/s")
print("="*60 + "\n")

Future - January

In [ ]:
import xarray as xr
import numpy as np
import os
import matplotlib.pyplot as plt
from windrose import WindroseAxes
from scipy.stats import wasserstein_distance, ks_2samp
import glob

%matplotlib inline

# --- 1. Configuration ---
MY_BINS = np.arange(0, 21, 3)
MAX_PERCENTAGE = 25
TARGET_HEIGHT = 150
YEARS = range(2040, 2081)
BASE_PATH = "/data/nobackup/boland/experiment2/WRF/run"

def calculate_metgrid_wind(file_list, target_z=TARGET_HEIGHT):
    u_150_all, v_150_all = [], []
    
    # Get center indices from first file
    with xr.open_dataset(file_list[0]) as temp_ds:
        mid_lat_idx = len(temp_ds.south_north) // 2
        mid_lon_idx = len(temp_ds.west_east) // 2

    for f in sorted(file_list):
        with xr.open_dataset(f) as ds:
            # Metgrid: UU, VV, GHT (Height in m)
            u_p = ds['UU'].isel(Time=0, south_north=mid_lat_idx, west_east_stag=mid_lon_idx).values
            v_p = ds['VV'].isel(Time=0, south_north_stag=mid_lat_idx, west_east=mid_lon_idx).values
            z_p = ds['GHT'].isel(Time=0, south_north=mid_lat_idx, west_east=mid_lon_idx).values
            
            # Vertical interpolation to target height
            u_150_all.append(np.interp(target_z, z_p, u_p))
            v_150_all.append(np.interp(target_z, z_p, v_p))
        
    u_arr, v_arr = np.array(u_150_all), np.array(v_150_all)
    speed = np.sqrt(u_arr**2 + v_arr**2)
    # Correct meteorological direction (0 is North)
    direction = (270 - np.rad2deg(np.arctan2(v_arr, u_arr))) % 360
    return speed, direction

# --- 2. Processing & Annual Gallery ---
all_speed, all_direction = [], []
yearly_data = {}

for y in YEARS:
    search_path = os.path.join(BASE_PATH, f"met_em.d01.{y}-01-*.nc")
    files = glob.glob(search_path)
    
    if files:
        print(f"Processing {y}...")
        speed, direction = calculate_metgrid_wind(files)
        yearly_data[y] = {'speed': speed, 'direction': direction}
        all_speed.extend(speed)
        all_direction.extend(direction)
        
        # Individual Annual Plot
        ax = WindroseAxes.from_ax()
        ax.bar(direction, speed, normed=True, bins=MY_BINS, opening=0.8, edgecolor='white')
        ax.set_ylim(0, MAX_PERCENTAGE)
        ax.set_legend(title="Wind Speed (m/s)", loc='lower right', decimal_places=0)
        ax.set_title(f"Year: August {y} | Mean: {np.mean(speed):.2f} m/s")
        plt.show()

# --- 3. Double Statistical Ranking (Vector-Based) ---

# Convert lists to numpy arrays for math
all_speed_arr = np.array(all_speed)
all_dir_arr = np.array(all_direction)

# Calculate "Full Climatology" components ONCE
# We use radians for trig functions
u_all = all_speed_arr * np.cos(np.deg2rad(all_dir_arr))
v_all = all_speed_arr * np.sin(np.deg2rad(all_dir_arr))

ks_scores = {}
emd_scores = {}

for y, data in yearly_data.items():
    s_yr = np.array(data['speed'])
    d_yr = np.array(data['direction'])
    
    # Calculate current year components
    u_yr = s_yr * np.cos(np.deg2rad(d_yr))
    v_yr = s_yr * np.sin(np.deg2rad(d_yr))

    # EMD Test (Wasserstein): Sum of distances for U and V
    emd_u = wasserstein_distance(u_all, u_yr)
    emd_v = wasserstein_distance(v_all, v_yr)
    emd_scores[y] = emd_u + emd_v
    
    # KS Test: Sum of D-statistics for U and V
    ks_u, _ = ks_2samp(u_all, u_yr)
    ks_v, _ = ks_2samp(v_all, v_yr)
    ks_scores[y] = ks_u + ks_v

# Find the years with the minimum discrepancy
best_ks_year = min(ks_scores, key=ks_scores.get)
best_emd_year = min(emd_scores, key=emd_scores.get)

# --- 4. Final Comparison Plots ---
def plot_comparison(winner_year, test_name, data_dict):
    fig = plt.figure(figsize=(14, 6))
    
    # Left Plot: The "Ideal" Distribution (40-year Climatology)
    ax1 = fig.add_subplot(121, projection="windrose")
    ax1.bar(all_direction, all_speed, normed=True, bins=MY_BINS, opening=0.8, edgecolor='white')
    ax1.set_ylim(0, MAX_PERCENTAGE)
    ax1.set_legend(title="Wind Speed (m/s)", loc='lower right', decimal_places=0)
    ax1.set_title(f"Target Climatology (2040 - 2080)", fontsize=12)
    
    # Right Plot: The "Winner" Year
    ax2 = fig.add_subplot(122, projection="windrose")
    ax2.bar(data_dict[winner_year]['direction'], data_dict[winner_year]['speed'], 
            normed=True, bins=MY_BINS, opening=0.8, edgecolor='white')
    ax2.set_ylim(0, MAX_PERCENTAGE)
    ax2.set_legend(title="Wind Speed (m/s)", loc='lower right', decimal_places=0)
    ax2.set_title(f"Most Representative Year ({test_name}): {winner_year}", fontsize=12)
    
    plt.tight_layout()
    plt.show()

print(f"\n--- Results ---\nKS Winner: {best_ks_year}\nEMD Winner: {best_emd_year}")
plot_comparison(best_ks_year, "KS", yearly_data)
plot_comparison(best_emd_year, "EMD", yearly_data)

print("\n" + "="*60)
# Print means for validation
print(f"KS Winner Year ({best_ks_year}) Mean Speed: {np.mean(yearly_data[best_ks_year]['speed']):.2f} m/s")
print(f"EMD Winner Year ({best_emd_year}) Mean Speed: {np.mean(yearly_data[best_emd_year]['speed']):.2f} m/s")
print(f"40-Year Climatology Mean Speed: {np.mean(all_speed):.2f} m/s")
print("="*60 + "\n")